In [16]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
import numpy as np
import copy

In [17]:
dataset_npz_path = "/mnt/c/Users/omen/Desktop/IDS_masters/carchallenge_0313_robust.npz"
test_path        = "/mnt/c/Users/omen/Desktop/IDS_masters/carhacking_0313_robust.npz"

# 단일 TCN 가중치 저장 경로
model1_path = "/mnt/c/Users/omen/Desktop/IDS_masters/TCN_single_0313.pth"

epochs = 20
batch_size = 64

# 단일 TCN이 사용할 채널
INPUT_CH = [0, 1, 2, 3, 4, 5, 6, 7]
num_input = len(INPUT_CH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"실행 디바이스: {device}")

실행 디바이스: cuda


In [18]:
#################################
# 1. Dataset Load Class
#################################

class LoadDatset(Dataset):
    def __init__(self, tensor_X, tensor_y):
        # [수정] NumPy 배열이 들어올 경우를 대비한 변환 로직
        if isinstance(tensor_X, np.ndarray):
            tensor_X = torch.from_numpy(tensor_X).float()
        if isinstance(tensor_y, np.ndarray):
            tensor_y = torch.from_numpy(tensor_y).long()

        # [추가] NaN(결측치)을 0.0으로 안전하게 대체하여 학습 붕괴 방지
        self.X = torch.nan_to_num(tensor_X, nan=0.0)
        self.y = tensor_y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
data_np = np.load(dataset_npz_path)
X_np, y_np = data_np["X"], data_np["y"]

In [19]:
#################################
# 2. Causal Convolution Layer
#################################

class CausalConv1d(nn.Conv1d):
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super().__init__(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=0
        )
        self.left_pad = (kernel_size - 1) * dilation

    def forward(self, x):
        x = F.pad(x, (self.left_pad, 0))  # 왼쪽만 패딩
        return super().forward(x)


In [20]:
#################################
# 3. TCN
#################################

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernal_size=3,  dilation=1, dropout=0.1):
        super().__init__()

        self.conv1 = CausalConv1d(n_inputs, n_outputs, kernel_size=kernal_size, dilation=dilation)
        
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = CausalConv1d(n_outputs, n_outputs, kernel_size=kernal_size, dilation=dilation)
        
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.conv3 = CausalConv1d(n_outputs, n_outputs, kernel_size=kernal_size, dilation=dilation)
        
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(dropout)
        
      
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self.init_weights()

    def init_weights(self):
        
        def init_one(layer):
            # weight_norm이면 weight_orig가 진짜 파라미터
            w = getattr(layer, "weight_orig", None)
            if w is None:
                w = layer.weight
            nn.init.kaiming_normal_(w)

            if layer.bias is not None:
                nn.init.zeros_(layer.bias)

        # conv1, conv2가 무엇이든(래퍼든 상속이든) 일단 'Conv1d 파라미터 가진 최종 모듈'에 적용
        init_one(self.conv1)
        init_one(self.conv2)
        init_one(self.conv3)

        if self.downsample is not None:
            nn.init.kaiming_normal_(self.downsample.weight)
            if self.downsample.bias is not None:
                nn.init.zeros_(self.downsample.bias)
                
    def forward(self, x):
        # x: (B, C, L)
        out = self.conv1(x)          # CausalConv1d 안에서 패딩 + 오른쪽 잘라내기
        out = self.relu1(out)
        out = self.dropout1(out)

        out = self.conv2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        out = self.conv3(out)
        out = self.relu3(out)
        out = self.dropout3(out)

        res = x if self.downsample is None else self.downsample(x)
        # CausalConv1d가 길이를 유지하니까 따로 slice 안 해도 됨
        return self.relu(out + res)

In [21]:
#################################
# 4. TCN
#################################

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channel, kernel_size=3, dropout =0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channel)

        dilation = [1,2,4]

        for i in range(num_levels):
            dilation_size = dilation[i]
            in_channels = num_inputs if i == 0 else num_channel[i-1]
            out_channels = num_channel[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size,  dilation = dilation_size, dropout=dropout)]
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

In [22]:
#################################
# 5. TCN
#################################

class SeqIDS(nn.Module):
    def __init__(self, num_input, num_classes, dropout_rate=0.2):
        super(SeqIDS, self).__init__()

        self.tcn = TemporalConvNet(
            num_inputs=num_input,          # feature 개수
            num_channel=[32, 64, 128],  # 각 레벨 채널 수
            kernel_size=3,
            dropout=dropout_rate
        )

        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Conv1d(128, num_classes, kernel_size=1)

    def forward(self, x, return_attn=False):
        x = self.tcn(x)
        x = self.dropout(x)
        logits = self.classifier(x)

        if return_attn:
            return logits
        
        return logits

In [23]:
#################################
# 6. Check NAN
#################################

data_load = np.load(dataset_npz_path)
X_np, y_np= data_load["X"], data_load["y"]

### NAN 값 확인 ###
for i in range(X_np.shape[1]):
        feat_nan = np.isnan(X_np[:, i, :]).sum()
        print(f"Feature {i}의 NaN 개수: {feat_nan}")

        nan_count_X = np.isnan(X_np).sum()
        nan_count_y = np.isnan(y_np).sum()
        print(f"데이터 검사 결과:")
        print(f"   - X 내 NaN 개수: {nan_count_X}개")
        print(f"   - y 내 NaN 개수: {nan_count_y}개")
        if nan_count_X > 0:
            print("주의: 입력 데이터(X)의 NaN은 0.0으로 자동 대체되어 학습됩니다.")


inf_count = np.isinf(X_np).sum() # 무한대 체크 추가
print(f"🔍 X 내 inf 개수: {inf_count}개")
if inf_count > 0:
    print("경고: 데이터에 inf(무한대)가 포함되어 있습니다. 인코딩 스크립트를 수정하세요.")

Feature 0의 NaN 개수: 0
데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
Feature 1의 NaN 개수: 0
데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
Feature 2의 NaN 개수: 0
데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
Feature 3의 NaN 개수: 0
데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
Feature 4의 NaN 개수: 0
데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
Feature 5의 NaN 개수: 0
데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
Feature 6의 NaN 개수: 0
데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
Feature 7의 NaN 개수: 0
데이터 검사 결과:
   - X 내 NaN 개수: 0개
   - y 내 NaN 개수: 0개
🔍 X 내 inf 개수: 0개


In [24]:
# ==========================================
# Oversampling
# ==========================================

def oversample_attack_windows(X, y, normal_ratio=0.75, attack_multiplier=3, class_multipliers=None, random_state=42):
    """
    X: (N_windows, T, F)
    y: (N_windows, T)

    normal_ratio: Normal 윈도우를 얼마나 유지할지 (0.2이면 20%만 유지)
    attack_multiplier: 공격 윈도우를 몇 배로 복제할지
    """
    np.random.seed(random_state)

    N = X.shape[0]
    T = y.shape[1]

    # 1. 공격 패킷 개수 세기
    attack_counts = (y != 0).sum(axis=1)

    # 2. Normal / Attack 윈도우 분리
    idx_normal = np.where(attack_counts == 0)[0]
    idx_attack = np.where(attack_counts > 0)[0]

    print(f"[INFO] Normal 윈도우: {len(idx_normal)}, 공격 윈도우: {len(idx_attack)}")

    # 3. Normal 윈도우 undersample
    n_keep_normal = int(len(idx_normal) * normal_ratio)
    if n_keep_normal < 1:
        n_keep_normal = 1

    keep_normal_idx = np.random.choice(idx_normal, size=n_keep_normal, replace=False)
    print(f"[INFO] Normal 윈도우 {len(idx_normal)} → {n_keep_normal} (undersample)")

    oversampled_indices =  []

     # class별 multiplier 없으면 전체 동일하게 적용
    if class_multipliers is None:
        class_multipliers = {}

    # 공격 윈도우에서 개별 공격 클래스 탐색
    for atk_class in [1, 2, 3, 4]:  # DoS, Fuzzing, Replay, Spoofing 가정
        # 해당 클래스가 있는 윈도우만 추출
        idx_class = []
        for i in idx_attack:
            if atk_class in y[i]:      # 해당 윈도우 안에 그 공격이 포함되어 있으면
                idx_class.append(i)

        if len(idx_class) == 0:
            continue

        # 1) 기본 multiplier 적용
        multiplier = attack_multiplier

        # 2) 만약 class_multipliers에 override 값이 있으면 덮어쓰기
        if atk_class in class_multipliers:
            multiplier = class_multipliers[atk_class]

        # oversample 수행
        sampled = np.random.choice(idx_class, size=len(idx_class) * multiplier, replace=True)
        oversampled_indices.extend(sampled)

        print(f"[ATTACK {atk_class}] {len(idx_class)}개 → {len(sampled)}개  (x{multiplier})")
    oversampled_indices = np.array(oversampled_indices, dtype=int)
    # 5. 합치기
    final_idx = np.concatenate([keep_normal_idx, idx_attack, oversampled_indices])
    np.random.shuffle(final_idx)

    X_bal = X[final_idx]
    y_bal = y[final_idx]

    print(f"[RESULT] 전체 윈도우: {len(final_idx)}개 (원래 {N}개)")

    return X_bal, y_bal, final_idx

In [25]:
#################################
# 6. First TCN Train
#################################

full_dataset = LoadDatset(X_np, y_np)

#### Validation Split (80:20) ###
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size

### 시드 고정 ###
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

print(f"데이터 분할 완료: 학습 {train_size}개 / 검증 {val_size}개")

### Model and DataLoader ###
model1 = SeqIDS(num_input=num_input1, num_classes=5, dropout_rate=0.5).to(device)


train1_idx = np.array(train_dataset.indices)

X_train1 = X_np[train1_idx]
y_train1 = y_np[train1_idx]


### Oversampling ###
X_train_bal1, y_train_bal1, _ = oversample_attack_windows(
    X_train1, y_train1,
    normal_ratio=1,
    attack_multiplier=1,          # 기본은 1로 두고
    class_multipliers={
        2: 2,
        4: 2},     # spoofing만 6배 이런 식으로 주는 걸 권장
    random_state=42
)


train_ds1 = LoadDatset(X_train_bal1, y_train_bal1)
# train_loader = DataLoader(train_ds1, batch_size=64, shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

### Config Model ###
W_conv1 = model1.tcn.network[0].conv1.weight
other_params = [p for n, p in model1.named_parameters() if "tcn.network.0.conv1.weight" not in n]

optimizer1 = torch.optim.Adam([
    {"params": [W_conv1], "weight_decay": 0},      # conv1은 수동 페널티를 위해 WD 0으로 설정
    {"params": other_params, "weight_decay": 1e-4} # 나머지는 일반적인 WD 적용
    ], lr=1e-4)

scheduler1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer1, mode='min', factor=0.5, patience=3
)



### data Weight AND LOSS Func ###
weights = torch.tensor([1.0, 1.0, 2.0, 1.0, 2.0]).to(device)
criterion1 = nn.CrossEntropyLoss(weight=weights, ignore_index=3)

### weight decay ###

# 15개 피처에 맞춘 추천 decay_map (Cell 93, 94 공통)
decay_map = {
    0: 0.01,    # ID IAT
    1: 0.01,  # Is Zero ID (중요)
    2: 0.01,    # Raw Entropy
    3: 0.01,   # Hamming Rate (중요)
    4: 0.01
}


def channel_l2_penalty(conv1_weight, decay_map):
    # conv1_weight: (out_ch, in_ch, k)
    pen = 0.0
    for ch, wd in decay_map.items():   # decay_map: {0:...,1:...,...}
        w_ch = conv1_weight[:, ch:ch+1, :]
        pen = pen + wd * (w_ch.pow(2).sum())
    return pen


# [일반화 5] Early Stopping 변수
best_val_loss = float('inf')
best_model_state = None

prev_train_loss = None
prev_val_loss   = None
overfit_wait    = 0
overfit_patience = 5
min_delta = 1e-4
es_wait   = 0   

print("\n Start Training...")

### Train Loop ###
for epoch in range(1, epochs + 1):
    model1.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        x1 = X_batch[:, Stage1_CH, :]   # 이름은 Stage1_CH 그대로 써도 됨
        logits1 = model1(x1)            # (B, 5, L)

        optimizer1.zero_grad()

        loss = criterion1(
            logits1.permute(0, 2, 1).reshape(-1, 5),
            y_batch.reshape(-1)
        )

        W = model1.tcn.network[0].conv1.weight
        loss = loss + channel_l2_penalty(W, decay_map)

        loss.backward()
        optimizer1.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    model1.eval()
    val_loss = 0
    with torch.no_grad():
        for X_val, y_val in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)

            x1v = X_val[:, Stage1_CH, :]
            logits1 = model1(x1v)

            loss = criterion1(
                logits1.permute(0, 2, 1).reshape(-1, 5),
                y_val.reshape(-1)
            )
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    scheduler1.step(avg_val_loss)
    current_lr = optimizer1.param_groups[0]['lr']

    print(f"Epoch [{epoch}/{epochs}] Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f} | LR: {current_lr:.6f}")
    ### Early Stopping  ###

    improved = (best_val_loss - avg_val_loss) > min_delta
    if improved:
        best_val_loss = avg_val_loss
        best_model_state = copy.deepcopy(model1.state_dict())
        torch.save(best_model_state, model1_path)
        es_wait = 0
    else:
        es_wait += 1
        print(f"  ⚠️ Val Loss 개선 안됨 (ES patience: {es_wait}/{overfit_patience})")
        if es_wait >= overfit_patience:
            print("🛑 Early Stopping 발동! 학습을 조기 종료합니다.")
            break

# 학습 종료 후, 가장 좋았던 모델 상태로 복구
if best_model_state is not None:
    model1.load_state_dict(best_model_state)
    print("\nBest Model 저장")

데이터 분할 완료: 학습 46899개 / 검증 11725개
[INFO] Normal 윈도우: 22043, 공격 윈도우: 24856
[INFO] Normal 윈도우 22043 → 22043 (undersample)
[ATTACK 1] 6951개 → 6951개  (x1)
[ATTACK 2] 5145개 → 10290개  (x2)
[ATTACK 3] 5247개 → 5247개  (x1)
[ATTACK 4] 7513개 → 15026개  (x2)
[RESULT] 전체 윈도우: 84413개 (원래 46899개)

 Start Training...
Epoch [1/20] Train Loss: 1.36726 | Val Loss: 0.58948 | LR: 0.000100
Epoch [2/20] Train Loss: 0.79515 | Val Loss: 0.32605 | LR: 0.000100
Epoch [3/20] Train Loss: 0.56072 | Val Loss: 0.16791 | LR: 0.000100
Epoch [4/20] Train Loss: 0.36951 | Val Loss: 0.09913 | LR: 0.000100
Epoch [5/20] Train Loss: 0.25562 | Val Loss: 0.06399 | LR: 0.000100
Epoch [6/20] Train Loss: 0.18512 | Val Loss: 0.05247 | LR: 0.000100
Epoch [7/20] Train Loss: 0.13906 | Val Loss: 0.04641 | LR: 0.000100
Epoch [8/20] Train Loss: 0.10602 | Val Loss: 0.04168 | LR: 0.000100
Epoch [9/20] Train Loss: 0.08188 | Val Loss: 0.03910 | LR: 0.000100
Epoch [10/20] Train Loss: 0.06514 | Val Loss: 0.03452 | LR: 0.000100
Epoch [11/20] Trai

In [26]:
#################################
# Single TCN Eval
#################################

model1 = SeqIDS(num_input=num_input1, num_classes=5, dropout_rate=0.5).to(device)

state1 = torch.load(model1_path, map_location=device)
model1.load_state_dict(state1)
model1.eval()

test_data = np.load(test_path)
X_np, y_np = test_data["X"], test_data["y"]

test_ds = LoadDatset(X_np, y_np)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

correct = 0
total = 0

num_classes = 5
conf_mat = torch.zeros(num_classes, num_classes, dtype=torch.int64)

with torch.no_grad():
    for input, labels in test_loader:
        input = input.to(device)
        labels = labels.to(device)

        x1 = input[:, Stage1_CH, :]
        logits = model1(x1)             # (B, 5, L)

        # Replay(3)는 예측 후보에서 제거
        logits[:, 3, :] = -1e9
        pred = logits.argmax(dim=1)     # 결과는 0,1,2,4 중 하나

        # Replay 정답 위치는 평가 제외
        valid_mask = (labels != 3)

        t_flat = labels[valid_mask].reshape(-1).cpu()
        p_flat = pred[valid_mask].reshape(-1).cpu()

        total += t_flat.numel()
        correct += (t_flat == p_flat).sum().item()

        for t, p in zip(t_flat, p_flat):
            conf_mat[t.long(), p.long()] += 1

accuracy = correct / total if total > 0 else 0.0

row_sum = conf_mat.sum(dim=1)
tp = conf_mat.diag()
fp = conf_mat.sum(dim=0) - tp
fn = row_sum - tp

precision_per_class = tp / (tp + fp + 1e-12)
recall_per_class    = tp / (tp + fn + 1e-12)
f1_per_class        = 2 * precision_per_class * recall_per_class / (precision_per_class + recall_per_class + 1e-12)

EVAL_CLASSES = [0, 1, 2, 4]
eval_idx = torch.tensor(EVAL_CLASSES)

precision_macro = precision_per_class[eval_idx].mean().item()
recall_macro    = recall_per_class[eval_idx].mean().item()
f1_macro        = f1_per_class[eval_idx].mean().item()

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision(macro, 0/1/2/4 only): {precision_macro:.4f}")
print(f"Recall(macro, 0/1/2/4 only)   : {recall_macro:.4f}")
print(f"F1(macro, 0/1/2/4 only)       : {f1_macro:.4f}")
print("Confusion Matrix:")
print(conf_mat)

LABEL_NAME = {0:"Normal", 1:"DoS", 2:"Fuzzing", 4:"Spoofing"}

print("\n=== Per-class Performance ===")
for i in EVAL_CLASSES:
    total_i = int(row_sum[i].item())
    correct_i = int(tp[i].item())
    acc_i = 100.0 * correct_i / total_i if total_i > 0 else 0.0
    print(f"{LABEL_NAME[i]:>10s} : {acc_i:6.2f}%  (correct {correct_i}/{total_i})")

Accuracy : 0.9314
Precision(macro, 0/1/2/4 only): 0.9663
Recall(macro, 0/1/2/4 only)   : 0.7677
F1(macro, 0/1/2/4 only)       : 0.7868
Confusion Matrix:
tensor([[28455591,        2,      625,        0,    19052],
        [       0,  1175042,        0,        0,        0],
        [   46176,        0,   937518,        0,        0],
        [       0,        0,        0,        0,        0],
        [ 2207144,      503,       73,        0,   296578]])

=== Per-class Performance ===
    Normal :  99.93%  (correct 28455591/28475270)
       DoS : 100.00%  (correct 1175042/1175042)
   Fuzzing :  95.31%  (correct 937518/983694)
  Spoofing :  11.84%  (correct 296578/2504298)
